In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Define your project directory path
PROJECT_PATH = '/content/drive/MyDrive/Implied-Volatility-project'

# Change the current working directory to your project folder
os.chdir(PROJECT_PATH)

# Verify you are in the right place
print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/drive/MyDrive/Implied-Volatility-project


In [3]:
import pandas as pd
import numpy as np

# Load our beautifully cleaned, structured files from Notebook 1
observed_df = pd.read_parquet("cleaned_observed.parquet")
predict_df  = pd.read_parquet("predict_template.parquet")

print(f"Notebook 2 Activated.")
print(f"Loaded Observed Pool: {observed_df.shape}")
print(f"Loaded Predict Template: {predict_df.shape}")

Notebook 2 Activated.
Loaded Observed Pool: (21840, 10)
Loaded Predict Template: (5460, 9)


In [4]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── 0. CONFIG ─────────────────────────────────────────────────────────────────

CLEANED_OBS_PATH     = "cleaned_observed.parquet"
PREDICT_PATH         = "predict_template.parquet"
FEATURES_OBS_PATH    = "features_observed.parquet"
FEATURES_PRED_PATH   = "features_predict.parquet"

DATETIME_COL         = "datetime"
TARGET_COL           = "iv"
STRIKE_COL           = "strike"
UNDERLYING_COL       = "underlying_price"
SYMBOL_COL           = "symbol"
OPTION_TYPE_COL      = "option_type"
EXPIRY_COL           = "expiry"

# Rolling window sizes (in number of timestamps, NOT calendar days)
ROLL_WINDOWS         = [3, 5, 10]      # short / medium / medium-long lookbacks
LAG_PERIODS          = [1, 2, 3]       # how many timestamps back to lag

In [5]:
# ── 1. LOAD ARTEFACTS ─────────────────────────────────────────────────────────

observed_df = pd.read_parquet(CLEANED_OBS_PATH)
predict_df  = pd.read_parquet(PREDICT_PATH)

# Ensure datetime dtype survived serialisation
observed_df[DATETIME_COL] = pd.to_datetime(observed_df[DATETIME_COL])
predict_df[DATETIME_COL]  = pd.to_datetime(predict_df[DATETIME_COL])
observed_df[EXPIRY_COL]   = pd.to_datetime(observed_df[EXPIRY_COL])
predict_df[EXPIRY_COL]    = pd.to_datetime(predict_df[EXPIRY_COL])

print(f"[LOAD] observed_df : {observed_df.shape}")
print(f"[LOAD] predict_df  : {predict_df.shape}")

[LOAD] observed_df : (21840, 10)
[LOAD] predict_df  : (5460, 9)


In [6]:
# ── 2. COMBINE FOR FEATURE ENGINEERING ───────────────────────────────────────
# We stack observed + predict so that rolling features on predict rows can
# look back into observed history.  The TARGET_COL of predict rows stays NaN
# throughout — it is never touched.

full_df = (
    pd.concat([observed_df, predict_df], ignore_index=True)
      .sort_values([DATETIME_COL, STRIKE_COL, OPTION_TYPE_COL])
      .reset_index(drop=True)
)

print(f"[COMBINE] Full stack shape : {full_df.shape}")

[COMBINE] Full stack shape : (27300, 10)


In [7]:
# ── 3. CROSS-SECTIONAL FEATURES  (per timestamp, no time-axis leakage) ────────
#
# At any timestamp t we observe the full volatility smile across strikes.
# All statistics below are computed WITHIN that smile only.

def add_cross_sectional_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each (datetime, option_type) slice — i.e. the call smile or put smile
    at a single point in time — compute:

    MONEYNESS FEATURES
    ──────────────────
    moneyness          : K / S  where K = strike, S = underlying_price
                         > 1 → OTM call / ITM put; < 1 → ITM call / OTM put
    log_moneyness      : log(K / S)
                         Linearises the strike dimension; the BS model and
                         SVI parameterisations are both natural in log-moneyness
                         space.  Values near 0 = ATM; negative = ITM call.
    moneyness_sq       : log(K/S)²  — captures the curvature ("smile" term)
                         in a simple quadratic expansion of the surface.

    STRIKE-DISTANCE FEATURES
    ─────────────────────────
    strike_dist_abs    : |K - S|  in raw index points
    strike_dist_norm   : |K - S| / S  — scale-free version; comparable
                         across different underlying price levels.

    ATM-RELATIVE FEATURES (cross-sectional normalisation)
    ──────────────────────────────────────────────────────
    iv_smile_mean      : mean IV across all strikes at this timestamp for
                         this option_type.  Represents the "level" of the smile.
    iv_smile_std       : std IV across strikes — "width" of the smile.
    iv_vs_smile_mean   : iv - iv_smile_mean  (only on observed rows)
                         Represents how far this strike's IV deviates from
                         the cross-sectional average.
    iv_smile_skew      : IV of lowest-strike minus IV of highest-strike,
                         divided by smile mean.  Positive → put skew dominates.
    iv_smile_zscore    : (iv - iv_smile_mean) / iv_smile_std  (observed only)

    All computed within (datetime, option_type) groups — zero lookahead.
    """
    df = df.copy()

    # 3a. Moneyness
    df["moneyness"]       = df[STRIKE_COL] / df[UNDERLYING_COL]
    df["log_moneyness"]   = np.log(df["moneyness"])
    df["moneyness_sq"]    = df["log_moneyness"] ** 2

    # 3b. Strike distance
    df["strike_dist_abs"]  = (df[STRIKE_COL] - df[UNDERLYING_COL]).abs()
    df["strike_dist_norm"] = df["strike_dist_abs"] / df[UNDERLYING_COL]

    # 3c. Cross-sectional smile statistics
    #     Group strictly by (datetime, option_type) — no time-axis aggregation
    grp_cs = df.groupby([DATETIME_COL, OPTION_TYPE_COL])

    cs_mean = grp_cs[TARGET_COL].transform("mean")
    cs_std  = grp_cs[TARGET_COL].transform("std").replace(0, np.nan)
    cs_min  = grp_cs[STRIKE_COL].transform("min")
    cs_max  = grp_cs[STRIKE_COL].transform("max")

    df["iv_smile_mean"]  = cs_mean
    df["iv_smile_std"]   = cs_std

    # iv at min-strike and max-strike for skew calculation
    def _skew(grp):
        lo_iv = grp.loc[grp[STRIKE_COL] == grp[STRIKE_COL].min(), TARGET_COL].mean()
        hi_iv = grp.loc[grp[STRIKE_COL] == grp[STRIKE_COL].max(), TARGET_COL].mean()
        smile_mean = grp[TARGET_COL].mean()
        skew_val   = (lo_iv - hi_iv) / smile_mean if smile_mean != 0 else np.nan
        grp["iv_smile_skew"] = skew_val
        return grp

    df = (
        df.groupby([DATETIME_COL, OPTION_TYPE_COL], group_keys=False)
          .apply(_skew)
    )

    # Deviation features — only meaningful where IV is observed
    df["iv_vs_smile_mean"] = df[TARGET_COL] - cs_mean
    df["iv_smile_zscore"]  = df["iv_vs_smile_mean"] / cs_std

    print(f"[CS FEATURES] Added: moneyness, log_moneyness, moneyness_sq, "
          f"strike_dist_abs, strike_dist_norm, iv_smile_mean, iv_smile_std, "
          f"iv_smile_skew, iv_vs_smile_mean, iv_smile_zscore")
    return df


full_df = add_cross_sectional_features(full_df)


[CS FEATURES] Added: moneyness, log_moneyness, moneyness_sq, strike_dist_abs, strike_dist_norm, iv_smile_mean, iv_smile_std, iv_smile_skew, iv_vs_smile_mean, iv_smile_zscore


In [8]:
# ──  VOLATILITY SURFACE & MONEYNESS ───────────────────────────────────────

def add_surface_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Moneyness (Distance between spot and strike)
    df['moneyness'] = df['underlying_price'] / df['strike']
    df['log_moneyness'] = np.log(df['underlying_price'] / df['strike'])



    print("[SURFACE] Added Moneyness and Cross-Sectional IV Rankings")
    return df

In [9]:
# ── 4. TIME-TO-EXPIRY FEATURE  (safe: only uses metadata, not IV values) ──────

def add_time_to_expiry(df: pd.DataFrame) -> pd.DataFrame:
    """
    dte_days : calendar days from datetime to expiry date.
               Options lose extrinsic value as DTE → 0 (theta decay).
               IV surfaces steepen near expiry for short-dated options.
    dte_sqrt : sqrt(DTE) — the natural time scaling in BSM (σ√T).
               Linearises the relationship between IV and time dimension.
    """
    df = df.copy()
    df["dte_days"] = (df[EXPIRY_COL] - df[DATETIME_COL]).dt.days.clip(lower=0)
    df["dte_sqrt"] = np.sqrt(df["dte_days"])

    df["is_expiry_day"] = (df["dte_days"] == 0).astype(int)

    print(f"[DTE] dte_days range : {df['dte_days'].min()} – {df['dte_days'].max()} days")
    return df


full_df = add_time_to_expiry(full_df)
full_df = add_surface_features(full_df)

[DTE] dte_days range : 0 – 19 days
[SURFACE] Added Moneyness and Cross-Sectional IV Rankings


In [10]:
# ── 5. ROLLING TIME-SERIES FEATURES  (strictly causal — shift before rolling) ──
#
# All rolling operations use .shift(1) FIRST, so the window at time t
# contains only t-1, t-2, … t-k.  This is the canonical way to prevent
# the current observation from leaking into its own feature.
#
# GroupBy key: (symbol, option_type)
#   → Each option contract's history stays isolated.
#   → Different strikes do NOT borrow each other's time-series history.

def add_rolling_features(
    df: pd.DataFrame,
    windows: list[int],
    lags: list[int],
) -> pd.DataFrame:
    """
    For every (symbol) group, sorted by datetime:

    LAG FEATURES
    ────────────
    iv_lag_{k}      : IV value k timestamps ago.
                      Captures mean-reversion and momentum in IV.

    ROLLING STATISTICS  (window = w timestamps, excluding current row)
    ─────────────────────────────────────────────────────────────────
    iv_roll_mean_{w} : rolling mean of past-w IV values.
                       Smoothed "local level" of volatility.
    iv_roll_std_{w}  : rolling std — local realised volatility-of-volatility.
    iv_roll_min_{w}  : rolling minimum — lower bound / support level.
    iv_roll_max_{w}  : rolling maximum — upper bound / resistance level.
    iv_roll_range_{w}: max - min over window — smile range width over time.

    VELOCITY & ACCELERATION
    ────────────────────────
    iv_velocity     : iv_lag_1 - iv_lag_2  (first difference of lagged IV)
                      Direction and speed of recent IV move.
    iv_accel        : (iv_lag_1 - iv_lag_2) - (iv_lag_2 - iv_lag_3)
                      Second difference — is the move accelerating?

    All features built from .shift(1) before rolling → zero lookahead.
    """
    df = df.copy().sort_values([SYMBOL_COL, DATETIME_COL])

    grp = df.groupby(SYMBOL_COL, group_keys=False)

    # ── 5a. Lag features ──────────────────────────────────────────────────────
    for k in lags:
        col = f"iv_lag_{k}"
        df[col] = grp[TARGET_COL].transform(lambda s: s.shift(k))

    # ── 5b. Rolling statistics (shift(1) bakes in the causal gap) ─────────────
    for w in windows:
        shifted = grp[TARGET_COL].transform(lambda s: s.shift(1))

        # We need to apply rolling per-group on the shifted series.
        # Strategy: attach shifted series temporarily, roll, then drop.
        df["_iv_shifted"] = shifted

        roll_grp = df.groupby(SYMBOL_COL)["_iv_shifted"]

        df[f"iv_roll_mean_{w}"]  = roll_grp.transform(
            lambda s: s.rolling(w, min_periods=1).mean()
        )
        df[f"iv_roll_std_{w}"]   = roll_grp.transform(
            lambda s: s.rolling(w, min_periods=2).std()
        )
        df[f"iv_roll_min_{w}"]   = roll_grp.transform(
            lambda s: s.rolling(w, min_periods=1).min()
        )
        df[f"iv_roll_max_{w}"]   = roll_grp.transform(
            lambda s: s.rolling(w, min_periods=1).max()
        )
        df[f"iv_roll_range_{w}"] = (
            df[f"iv_roll_max_{w}"] - df[f"iv_roll_min_{w}"]
        )

    df.drop(columns=["_iv_shifted"], inplace=True)

    # ── 5c. Velocity & acceleration (from already-computed lag cols) ──────────
    if "iv_lag_1" in df.columns and "iv_lag_2" in df.columns:
        df["iv_velocity"] = df["iv_lag_1"] - df["iv_lag_2"]
    if "iv_lag_2" in df.columns and "iv_lag_3" in df.columns:
        df["iv_accel"]    = (df["iv_lag_1"] - df["iv_lag_2"]) \
                          - (df["iv_lag_2"] - df["iv_lag_3"])

    roll_cols = (
        [f"iv_lag_{k}"        for k in lags] +
        [f"iv_roll_mean_{w}"  for w in windows] +
        [f"iv_roll_std_{w}"   for w in windows] +
        [f"iv_roll_min_{w}"   for w in windows] +
        [f"iv_roll_max_{w}"   for w in windows] +
        [f"iv_roll_range_{w}" for w in windows] +
        ["iv_velocity", "iv_accel"]
    )
    print(f"[ROLLING] Added {len(roll_cols)} features across "
          f"windows={windows}, lags={lags}")
    return df


full_df = add_rolling_features(full_df, ROLL_WINDOWS, LAG_PERIODS)

[ROLLING] Added 20 features across windows=[3, 5, 10], lags=[1, 2, 3]


In [11]:
# ── 6. INTERACTION FEATURES ───────────────────────────────────────────────────

def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lightweight multiplicative terms that the linear part of any model
    cannot represent on its own:

    moneyness_x_dte  : log_moneyness × dte_sqrt
                       Captures how the smile shape changes as expiry
                       approaches (near-expiry smiles steepen at the wings).
    roll_mean_x_mono : iv_roll_mean_5 × log_moneyness
                       How the local IV level interacts with where we are
                       on the smile — OTM wings tend to move more in
                       high-vol regimes.
    """
    df = df.copy()

    if "iv_roll_mean_5" in df.columns:
        df["moneyness_x_dte"]    = df["log_moneyness"] * df["dte_sqrt"]
        df["roll_mean_x_mono"]   = df["iv_roll_mean_5"] * df["log_moneyness"]

    print("[INTERACT] Added: moneyness_x_dte, roll_mean_x_mono")
    return df


full_df = add_interaction_features(full_df)


[INTERACT] Added: moneyness_x_dte, roll_mean_x_mono


In [12]:
# ── 7. SPLIT BACK INTO OBSERVED / PREDICT AND SAVE ───────────────────────────

def split_and_save(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Re-split the feature-enriched full stack back into:
      - features_obs  : observed rows (have TARGET_COL, used for training)
      - features_pred : predict rows  (TARGET_COL still NaN, for submission)
    """
    features_obs  = df[df[TARGET_COL].notna()].copy()
    features_pred = df[df[TARGET_COL].isna()].copy()

    features_obs.to_parquet(FEATURES_OBS_PATH,  index=False)
    features_pred.to_parquet(FEATURES_PRED_PATH, index=False)

    print(f"\n[SAVE] features_observed : {features_obs.shape}  → {FEATURES_OBS_PATH}")
    print(f"[SAVE] features_predict  : {features_pred.shape}  → {FEATURES_PRED_PATH}")
    return features_obs, features_pred


features_obs, features_pred = split_and_save(full_df)


[SAVE] features_observed : (21840, 45)  → features_observed.parquet
[SAVE] features_predict  : (5460, 45)  → features_predict.parquet


In [13]:
# ── 8. FEATURE SUMMARY (FIXED) ───────────────────────────────────────────────

def feature_summary(df: pd.DataFrame) -> None:
    """
    Print a clean table of all engineered feature columns:
    name | non-null count | null % | mean | std
    """
    # FIX APPLIED: Added "index" to the exclusion list so it doesn't try to compute math on text strings
    non_id_cols = [
        c for c in df.columns
        if c not in [DATETIME_COL, SYMBOL_COL, EXPIRY_COL, "index",
                     OPTION_TYPE_COL, TARGET_COL, "is_outlier", "time_idx"]
    ]

    stats = df[non_id_cols].agg(["count", "mean", "std"]).T
    stats["null_%"] = ((len(df) - stats["count"]) / len(df) * 100).round(1)
    stats = stats[["count", "null_%", "mean", "std"]].round(4)
    print("\n[FEATURE SUMMARY]")
    print(stats.to_string())


feature_summary(features_obs)


[FEATURE SUMMARY]
                    count  null_%        mean       std
underlying_price  21840.0     0.0  25563.4397  318.0734
strike            21840.0     0.0  25152.3810  807.0003
moneyness         21840.0     0.0      1.0174    0.0351
log_moneyness     21840.0     0.0      0.0166    0.0345
moneyness_sq      21840.0     0.0      0.0015    0.0017
strike_dist_abs   21840.0     0.0    795.3092  538.8814
strike_dist_norm  21840.0     0.0      0.0311    0.0209
iv_smile_mean     21840.0     0.0      0.1829    0.1936
iv_smile_std      21840.0     0.0      0.0443    0.0979
iv_smile_skew     14229.0    34.8      0.2437    0.5195
iv_vs_smile_mean  21840.0     0.0      0.0000    0.1025
iv_smile_zscore   21840.0     0.0     -0.0000    0.9543
dte_days          21840.0     0.0      9.8330    5.9412
dte_sqrt          21840.0     0.0      2.9016    1.1890
is_expiry_day     21840.0     0.0      0.0775    0.2673
iv_lag_1          17490.0    19.9      0.1806    0.1994
iv_lag_2          17418.0    

In [14]:
# ── 9. LOOKAHEAD AUDIT ────────────────────────────────────────────────────────

def audit_lookahead(df: pd.DataFrame) -> None:
    """
    Verify the no-lookahead contract for rolling features:
      For each symbol, the iv_lag_1 at time t must equal the raw IV at t-1.
      Any mismatch → shift() was not applied correctly.
    """
    if "iv_lag_1" not in df.columns:
        print("[AUDIT] iv_lag_1 not found — skipping.")
        return

    sample_symbol = df[SYMBOL_COL].value_counts().idxmax()
    sub = (
        df[df[SYMBOL_COL] == sample_symbol]
          [[DATETIME_COL, TARGET_COL, "iv_lag_1"]]
          .sort_values(DATETIME_COL)
          .reset_index(drop=True)
    )

    mismatches = 0
    for i in range(1, len(sub)):
        expected_lag = sub.loc[i - 1, TARGET_COL]
        actual_lag   = sub.loc[i,     "iv_lag_1"]
        if pd.notna(expected_lag) and pd.notna(actual_lag):
            if not np.isclose(expected_lag, actual_lag, atol=1e-8):
                mismatches += 1

    if mismatches == 0:
        print(f"\n[AUDIT] ✓ Lookahead check passed on symbol '{sample_symbol}': "
              f"iv_lag_1[t] == iv[t-1] for all observed rows.")
    else:
        print(f"\n[AUDIT] ✗ {mismatches} mismatches found — review shift logic!")


audit_lookahead(features_obs)



[AUDIT] ✓ Lookahead check passed on symbol 'NIFTY27JAN2624700PE': iv_lag_1[t] == iv[t-1] for all observed rows.
